In [6]:
import pandas as pd

In [7]:
df = pd.read_csv("abt_churn.csv")

In [8]:
df.head()

,dtRef,idUsuario,qtdeTransacoes,qtdeDias,mediaTransacoesDias,saldoPontos,qtdePontosPos,qtdePontosNeg,qtdeDiasUltimaTransacao,qtdeDiasPrimeiraTransacao,...,saldoPontosD28,qtdePontosPosD28,qtdePontosNegD28,propAvgQtdeTransacoes,propAvgQtdeDias,propAvgMediaTransacoesDias,propAvgSaldoPontos,propAvgQtdePontosPos,propAvgQtdePontosNeg,flagChurn
0,2024-06-01,000ff655-fa9f-4baa-a108-47f581ec52a1,266,27,9.851852,635,2635,-2000,1.0,89.0,...,151,151,0,3.889781,3.278281,2.135602,1.516314,4.101926,8.944444,1
1,2024-10-01,000ff655-fa9f-4baa-a108-47f581ec52a1,268,28,9.571429,686,2686,-2000,4.0,211.0,...,51,51,0,3.309865,2.567615,2.328737,1.288278,2.920954,5.167037,1
2,2024-04-01,000ff655-fa9f-4baa-a108-47f581ec52a1,188,11,17.090909,275,1275,-1000,3.0,28.0,...,275,1275,-1000,3.805468,1.820201,3.739048,0.798622,2.634551,7.162791,0
3,2024-05-01,000ff655-fa9f-4baa-a108-47f581ec52a1,262,24,10.916667,484,2484,-2000,2.0,58.0,...,107,1107,-1000,4.246216,3.255244,2.322640,1.275749,4.275765,9.922414,0
4,2024-08-01,001749bd-37b5-4b1e-8111-f9fbba90f530,1,1,1.000000,50,50,0,21.0,21.0,...,50,50,0,0.013317,0.103973,0.236635,0.104533,0.063302,0.000000,1


## 1. S - Sample (Amostragem)
O primeiro passo é garantir que o modelo seja treinado e validado em amostras
justas e representativas, evitando o vazamento de dados (data leakage) e
garantindo a estabilidade temporal.
Sendo necessário:
- Out-of-Time (OOT): Separe a safra de dados mais recente (o último mês
disponível na base) e reserve-a. Essa base não deve ser tocada até o final
do projeto; ela servirá para testar a estabilidade do modelo no tempo.
- Train/Test Split: Com o restante dos dados, faça a divisão clássica entre
base de Treino (ex: 80%) e base de Teste (ex: 20%).
- Estratificação: Ao fazer o split, garanta que a proporção da variável
resposta (flag de Churn) seja idêntica tanto no Treino quanto no Teste.

In [9]:
# Analisando ocorrências das datas
df["dtRef"].value_counts().sort_index(ascending=False)

dtRef
2025-04-01    303
2025-03-01    451
2025-02-01    374
2025-01-01    194
2024-12-01    198
2024-11-01    258
2024-10-01    349
2024-09-01    371
2024-08-01    415
2024-07-01    422
2024-06-01    426
2024-05-01    542
2024-04-01    528
2024-03-01    665
Name: count, dtype: int64

Não há nenhum registro que ocorreu fora do primeiro dia de cada mês, sendo possível apenas isolar os registros da data "2025-04-01"

In [10]:
# Isolando o último mês mais recente disponível na base
df_oot = df[df["dtRef"] == "2025-04-01"].copy()
df_model = df[df["dtRef"] != "2025-04-01"].copy()

# Divisão de variáveis (Features vs Target)
X = df_model.drop(columns=["flagChurn"])
y = df_model["flagChurn"]

from sklearn.model_selection import train_test_split

# Split com Estratificação
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)